## Reads the PID log messages and builds a dictionary with the results

Craig Lage
05-Apr-26

In [ ]:
# Times Square Parameters
day_obs = 20260405
store_dir = "/home/c/cslage/u/MTAOS/times_square_reports/" # Where to store the results

In [ ]:
import numpy as np
import pickle as pkl
import re
import matplotlib.pyplot as plt
import lsst.summit.utils.butlerUtils as butlerUtils
from lsst.summit.utils.efdUtils import makeEfdClient, getEfdData, getMostRecentRowWithDataBefore
from lsst.summit.utils.butlerUtils import getExpRecordFromDataId

## Extract the pidDict for one day_obs

In [ ]:
exposureList = []
for record in butler.registry.queryDimensionRecords("exposure", 
                    where=f"exposure.day_obs={day_obs} and instrument='LSSTCam'"):
    exposureList.append([record.id, record])
exposureList.sort(key=lambda x: x[0])

pidDict = {}
for n, exposure in enumerate(exposureList):
    if n == 0:
        continue
    [expId, record] = exposure
    seqNum = int(expId - 1E5 * day_obs)
    [expIdm1, recordm1] = exposureList[n-1]
    pidData = getEfdData(client, 
                         topic='lsst.sal.MTAOS.logevent_logMessage',
                         columns=['message'],
                         begin=recordm1.timespan.end,
                         end=record.timespan.end)
    if len(pidData) == 0:
        continue
    FoundPIDMessage = False
    for msg in pidData['message'].values:
        if 'PID' in msg:
            matches = re.findall(r'\[([^\]]+)\]', msg, re.DOTALL)
            arrays = [np.fromstring(m, sep=' ') for m in matches]
            corr0 = arrays[3][0]
            FoundPIDMessage = True
    if not FoundPIDMessage:
        continue
    # Now loop through to find the matching dof_event
    for [sub_expId, sub_record] in exposureList:
        thisSeqNum = int(sub_expId - 1E5 * day_obs)
        if thisSeqNum < seqNum - 4 or thisSeqNum >= seqNum:
            continue
        dof_event = getMostRecentRowWithDataBefore(
            client,
            "lsst.sal.MTAOS.logevent_degreeOfFreedom",
            timeToLookBefore=record.timespan.end,
        )
        if len(dof_event) == 0:
            continue
        visitId = dof_event['visitId']
        visit0 = dof_event['visitDoF0'] # This needs to match the Correction[0].
        if abs(-corr0 - visit0) < 1E-9:
            break
    
    kp_array = np.array([dof_event[f'kpGain{i:d}'] for i in range(22)])
    ki_array = np.array([dof_event[f'kiGain{i:d}'] for i in range(22)])
    kd_array = np.array([dof_event[f'kdGain{i:d}'] for i in range(22)])
    thisDict={"Source": visitId,
              "Error": arrays[0],
              "I": arrays[1],
              "Clipped_I": arrays[2],
              "Correction": arrays[3],
              "Kp": kp_array,
              "Ki": ki_array,
              "Kd": kd_array}            
    pidDict[expId] = thisDict
    if expId % 50 == 0:
        print(f"Finished {expId}.")
filename = f"/home/c/cslage/u/MTAOS/times_square_reports/PID_data_{day_obs}.pkl"
with open(filename, 'wb') as f:
    pkl.dump(pidDict, f)


## Example dictionary entry

In [ ]:
seqNum = 100
expId = int(1E5 * day_obs + seqNum)
print(expId)
thisDict = pidDict[expId]
for key in thisDict.keys():
    print(key, thisDict[key])
    

In [ ]:
day_obs = 20260404
filename = f"/home/c/cslage/u/MTAOS/times_square_reports/PID_data_{day_obs}.pkl"
with open(filename, 'rb') as f:
    pidDict = pkl.load(f)


In [ ]:
seqNum = 143
expId = int(1E5 * day_obs + seqNum)
print(expId)
thisDict = pidDict[expId]
for key in thisDict.keys():
    print(key, thisDict[key])
    

In [ ]:
for expId in pidDict.keys():
    thisDict = pidDict[expId]
    print(expId, thisDict['Kp'][0], thisDict['Ki'][0])
